# HMM-Diffusion-EW

Regime-conditional synthetic financial time series with **equal-weight** true regime labels.

Each of the ten assets is z-scored, then averaged with equal weight. Vol_Regime is run on that series over the full 10-asset overlap, **2001-01-01 to 2022-08-31**. The HMM emission is the same equal-weight series. The HMM is trained on **2001-01-01 to 2014-01-03**. **2014-01-06 to 2022-08-31** is held out as the test set. There is no validation slice.

Diffusion specialists still generate 10-asset windows, labeled by the EW regimes, and are trained only on the 2001–2014 dates.

Run the EW scripts before this notebook:

```
python scripts/01_label_regimes_ew.py
python scripts/02_build_diffusion_dataset_ew.py
python scripts/03_train_specialists_ew.py
python scripts/04_generate_pools_ew.py
```

Without stages 3 and 4 the notebook uses placeholder pools resampled from real EW returns. This notebook does not load `configs/default.yaml` and does not touch the A001 artifacts.


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from hmmdiff.config import bootstrap_imports, config_path, load_config

cfg = load_config("configs/ew.yaml")
bootstrap_imports()

from hmmdiff import ew, models, plots, pools, regimes, stitch

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

N_REGIMES = cfg["regimes"]["n_regimes"]
TOP_K = cfg["hmm"]["top_k"]
REFERENCE = cfg["reference"]

print(f"{N_REGIMES} regimes, seq_len {cfg['diffusion']['seq_len']}")
print(f"config {Path('configs/ew.yaml')}")


## Data preparation

Ten-asset log returns start when every name has a price (A014 binds the start at 2001-01-01). Each column is standardized, the equal-weight average is z-scored, and dates inherit the A001 train/test cutoff: train through 2014-01-03, test from 2014-01-06. The HMM uses the full train slice; there is no inner train/validation split.


In [ ]:
returns = ew.load_ew_returns(cfg)
counts = ew.split_counts(returns)
train_series, test_series = ew.split_series(returns)

print(f"returns      {counts['n_returns']:>6}  {counts['start_date']} to {counts['end_date']}")
print(f"train / test {counts['n_train']:>6} / {counts['n_test']}")
print(f"train end    {counts['train_end_date']}    test start {counts['test_start_date']}")

assert counts["n_returns"] == REFERENCE["n_returns"]
assert counts["n_train"] == REFERENCE["n_train"]
assert counts["n_test"] == REFERENCE["n_test"]
assert counts["start_date"] == REFERENCE["start_date"]
assert counts["train_end_date"] == REFERENCE["train_end_date"]
assert counts["test_start_date"] == REFERENCE["test_start_date"]

plots.plot_prices(returns["z_return"].to_numpy(dtype=float), title="Equal-weight standardized return")


## Volatility regimes

Labels are loaded from the cache written by `scripts/01_label_regimes_ew.py`. Clustering uses the **full** 2001–2022 equal-weight series. Labels are reordered so regime index increases with variance. Plots below show the full labeled series; the HMM is fit only on the train dates.


In [ ]:
labels = regimes.load_labels(config_path(cfg, "regime_labels"))
regime_labels = labels.labels.astype(int)
train_labels, test_labels = ew.split_labels(returns, regime_labels)
segment_colors = labels.segment_colors()

print(f"labels {len(regime_labels)} days  train {len(train_labels)}  test {len(test_labels)}")
display(plots.regime_moments(returns["z_return"].to_numpy(dtype=float), regime_labels, N_REGIMES))

plots.plot_regime_spans(
    returns["z_return"].to_numpy(dtype=float),
    labels.changepoints,
    segment_colors,
    title="EW Returns & Volatility Regimes (2001–2022)",
)
plots.plot_regime_spans(
    returns["z_return"].to_numpy(dtype=float),
    labels.changepoints,
    segment_colors,
    overlay=regimes.ewm_volatility(returns["z_return"].to_numpy(dtype=float), cfg["regimes"]["ewm_com"]),
    title="EW Returns, EWMA Volatility & Regimes",
)
plots.plot_regime_labels(regime_labels)


## Model frames and transition structure

`train_data` is 2001–2014 and `test_data` is 2014–2022. Each frame holds the EW regime label, the EW emission, and the lagged emission for the Markov switching variant. The empirical transition matrix below is estimated on the train labels, which is the path the HMM is trained against.


In [ ]:
train_data, test_data = ew.build_train_test_frames(returns, regime_labels)

coverage = pd.DataFrame(
    {
        "train": np.bincount(train_data.regime.values, minlength=N_REGIMES),
        "test": np.bincount(test_data.regime.values, minlength=N_REGIMES),
    },
    index=pd.Index(range(N_REGIMES), name="regime"),
)
display(coverage)

missing_train = coverage.index[coverage["train"] == 0].tolist()
assert not missing_train, (
    f"Regimes {missing_train} are absent from the training slice, so their emission parameters "
    "would come from the prior. Re-run scripts/01_label_regimes_ew.py with a different "
    "regimes.changepoint_penalty."
)
missing_test = coverage.index[coverage["test"] == 0].tolist()
if missing_test:
    print(
        f"Note: regimes {missing_test} do not occur in the test slice, so test accuracy "
        "is measured over the remaining regimes only."
    )


In [ ]:
empirical_transmat = regimes.empirical_transition_matrix(train_labels, N_REGIMES)
outbound_transmat = regimes.outbound_transition_matrix(empirical_transmat, N_REGIMES)

index = pd.Index(range(N_REGIMES), name="from")
columns = pd.Index(range(N_REGIMES), name="to")

print("Empirical transition matrix (train)")
display(pd.DataFrame(empirical_transmat, index=index, columns=columns))

print("Outbound transition matrix (diagonal removed, rows renormalized)")
display(pd.DataFrame(outbound_transmat, index=index, columns=columns))

print("Average consecutive days per visit (full 2001–2022 labels)")
display(pd.Series(regimes.average_regime_length(regime_labels)).sort_index())


## Diffusion pools

One pool of synthetic 128-day, 10-asset windows per regime. For stitching, each window is mapped to EW emission units: z-score each asset with the overlap moments, average, then z-score the equal-weight series. If specialists are not trained yet, placeholder pools are resampled from real per-regime EW train returns.


In [ ]:
pools_root = config_path(cfg, "pools")
USING_PLACEHOLDER = not pools.pools_available(pools_root, N_REGIMES)
GENERATED_TITLE = (
    "PLACEHOLDER: Resampled Real EW Returns (not diffusion)"
    if USING_PLACEHOLDER
    else "HMM Diffusion Generated EW Returns"
)

if USING_PLACEHOLDER:
    print(
        "WARNING: no trained specialist pools found under "
        f"{pools_root.relative_to(REPO_ROOT)}.\n"
        "Falling back to placeholder pools resampled from the real per-regime EW returns.\n"
        "Run scripts/03_train_specialists_ew.py and scripts/04_generate_pools_ew.py for real results."
    )
    generated_images = pools.placeholder_pools(
        train_series, train_labels, N_REGIMES, size=len(train_series), seed=0
    )
else:
    regime_stats = pools.regime_emission_stats(
        train_data.emission.values, train_data.regime.values, N_REGIMES
    )
    generated_images = pools.load_generated_images_ew(
        pools_root,
        N_REGIMES,
        scale=labels.metadata,
        calibrate_regimes=regime_stats,
    )
    print(
        "Pool EW rescaled: 10-asset z-score, equal-weight, then EW z-score "
        f"(ew_mean={labels.metadata['ew_mean']:.6e}, ew_sd={labels.metadata['ew_std']:.6e}) "
        "+ per-regime calibration"
    )
    for k in range(N_REGIMES):
        meta = pools.load_pool_meta(pools_root, k)
        print(f"regime {k}: {meta.get('n_pool')} windows from {Path(meta.get('checkpoint_dir', '')).name}")

pool_summary = pd.DataFrame(
    {
        "pool_length": [generated_images[str(k)].size for k in range(N_REGIMES)],
        "pool_mean": [generated_images[str(k)].mean() for k in range(N_REGIMES)],
        "pool_var": [generated_images[str(k)].var() for k in range(N_REGIMES)],
        "real_mean": [train_series[train_labels == k].mean() for k in range(N_REGIMES)],
        "real_var": [train_series[train_labels == k].var() for k in range(N_REGIMES)],
    },
    index=pd.Index(range(N_REGIMES), name="regime"),
)
display(pool_summary)

candidate_paths = [train_data.regime.values, test_data.regime.values]
problems, _ = pools.check_pools(
    generated_images, candidate_paths, train_series, train_labels, N_REGIMES
)
for problem in problems:
    print(f"POOL CHECK: {problem}")
if not problems:
    print("Pool checks passed: finite, sufficient for every regime path, variance ordering matches.")


## HMM variants

Four ways of estimating a transition matrix and per-regime Gaussian emission parameters. Supervised HMM treats regimes and emissions as observed and is fit with NUTS. Markov switching regresses the emission mean on the previous emission. Semi-supervised keeps labels on the first half of training and treats the second half as unlabeled. Neural HMM uses a small network for emission parameters fit by SVI.

All four are evaluated the same way: forward-backward smoothing, comparison to true EW labels, stitching along the estimated path, and per-regime moment tables. Accuracy is reported on train (2001–2014) and test (2014–2022). Top-2 accuracy is reported alongside top-1 because adjacent volatility regimes overlap heavily.


In [ ]:
init_dist = models.initial_distribution(train_data, N_REGIMES)


def evaluate_variant(key, fit):
    """Fit diagnostics, state estimation, stitching, and plots for one HMM variant."""
    print("=" * 78)
    print(fit.name)
    print("=" * 78)

    print("Posterior transition matrix")
    display(pd.DataFrame(fit.transmat.round(3), index=index, columns=columns))
    print("Emission parameters")
    display(fit.summary())

    train_probs, train_est = models.estimate_states(
        fit, train_data.emission.values, init_dist, N_REGIMES
    )
    test_probs, test_est = models.estimate_states(
        fit, test_data.emission.values, init_dist, N_REGIMES
    )
    train_acc = models.accuracy_report(
        train_data.regime.values, train_est, train_probs, N_REGIMES, TOP_K
    )
    test_acc = models.accuracy_report(
        test_data.regime.values, test_est, test_probs, N_REGIMES, TOP_K
    )

    print(
        f"Train accuracy {train_acc['accuracy']:>7.2%}   "
        f"top-{TOP_K} {train_acc[f'top_{TOP_K}_accuracy']:>7.2%}"
    )
    print(
        f"Test  accuracy {test_acc['accuracy']:>7.2%}   "
        f"top-{TOP_K} {test_acc[f'top_{TOP_K}_accuracy']:>7.2%}"
    )

    plots.plot_estimates(train_est, train_data.regime.values, f"{fit.name}: Train")
    plots.plot_estimates(test_est, test_data.regime.values, f"{fit.name}: Test")

    oracle = stitch.stitch(generated_images, train_data.regime.values)
    estimated, test_estimated = plots.plot_paper_train_test(
        train_data.emission.values,
        train_data.regime.values,
        train_est,
        test_data.emission.values,
        test_data.regime.values,
        test_est,
        generated_images,
        generated_title=GENERATED_TITLE,
    )

    table = stitch.backtest_table(
        train_data.emission.values,
        train_data.regime.values,
        estimated,
        train_est,
        N_REGIMES,
    )
    test_table = stitch.backtest_table(
        test_data.emission.values,
        test_data.regime.values,
        test_estimated,
        test_est,
        N_REGIMES,
    )
    print("Train backtest")
    display(table)
    print("Test backtest")
    display(test_table)

    return {
        "name": fit.name,
        "fit": fit,
        "train_accuracy": train_acc["accuracy"],
        "train_top_k": train_acc[f"top_{TOP_K}_accuracy"],
        "test_accuracy": test_acc["accuracy"],
        "test_top_k": test_acc[f"top_{TOP_K}_accuracy"],
        "oracle": oracle,
        "estimated": estimated,
        "test_estimated": test_estimated,
        "backtest": table,
        "test_backtest": test_table,
    }


results = {}


In [ ]:
results["supervised"] = evaluate_variant(
    "supervised", models.fit_supervised_hmm(train_data, N_REGIMES, cfg)
)


In [ ]:
results["markov_switching"] = evaluate_variant(
    "markov_switching", models.fit_markov_switching(train_data, N_REGIMES, cfg)
)


In [ ]:
results["semi_supervised"] = evaluate_variant(
    "semi_supervised", models.fit_semi_supervised_hmm(train_data, N_REGIMES, cfg)
)


In [ ]:
neural_fit = models.fit_neural_hmm(train_data, N_REGIMES, cfg)
plots.plot_elbo(neural_fit.diagnostics["elbo_losses"])
results["neural"] = evaluate_variant("neural", neural_fit)


## Results

State-estimation accuracy for all four variants, then the per-regime distributional fit of the generated series. The moment comparison is grouped by the true EW regime while generated values follow the estimated path. The oracle column stitches along the true path to isolate generator quality.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "variant": r["name"],
            "train": r["train_accuracy"],
            f"train_top{TOP_K}": r["train_top_k"],
            "test": r["test_accuracy"],
            f"test_top{TOP_K}": r["test_top_k"],
        }
        for key, r in results.items()
    ]
).set_index("variant")

display(summary.style.format("{:.2%}"))


In [ ]:
best_key = summary["test"].idxmax()
best = next(r for r in results.values() if r["name"] == best_key)

oracle_table = stitch.backtest_table(
    train_data.emission.values,
    train_data.regime.values,
    best["oracle"],
    train_data.regime.values,
    N_REGIMES,
)

comparison = pd.DataFrame(
    {
        "real_var": oracle_table["real_var"],
        "oracle_var": oracle_table["gen_var"],
        "hmm_var": best["backtest"]["gen_var"],
        "real_mean": oracle_table["real_mean"],
        "oracle_mean": oracle_table["gen_mean"],
        "hmm_mean": best["backtest"]["gen_mean"],
    }
)

print(f"Per-regime moments, stitched with {best_key}")
display(comparison)

if USING_PLACEHOLDER:
    print(
        "\nThese generated columns come from placeholder resampling, not the diffusion specialists."
    )
